In [ ]:
import phasespace
import numpy as np

import pandas as pd

In [ ]:
MASS_PARENT = 100 # GeV/c^2
MASS_CHILD = 10   # GeV/c^2

pmag = 500 # GeV/c

nevents_to_generate = 10

boost_vector = np.array([0,0, pmag, np.sqrt(pmag**2 + MASS_PARENT**2)])
boost_vectors = np.tile(boost_vector, (nevents_to_generate,1))

print("Created the boost vector and tiled it so that we have a boost vector for each decay we are simulating")
print()
print(boost_vectors)
print()

print(f"Generating {nevents_to_generate} decays")

weights, particles = phasespace.nbody_decay(MASS_PARENT, \
                                            [MASS_CHILD, MASS_CHILD]).generate(n_events=nevents_to_generate, boost_to=boost_vectors)

print("Generated the decays!")
print()
print(particles)

In [ ]:
import numpy as np

def ray_cylinder_intersection(
    origins: np.ndarray,      # Shape (N, 3) - starting points
    directions: np.ndarray,   # Shape (N, 3) - momentum/direction vectors (not necessarily normalized)
    radius: float,            # Cylinder radius
    half_length: float        # Half the cylinder length (extends from -half_length to +half_length on z-axis)
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute entry and exit points for rays intersecting a cylinder centered at origin,
    aligned along the z-axis.
    
    Parameters
    ----------
    origins : ndarray of shape (N, 3)
        Starting points of the rays (outside the cylinder)
    directions : ndarray of shape (N, 3)
        Direction vectors (momentum vectors) of the rays
    radius : float
        Radius of the cylinder
    half_length : float
        Half-length of the cylinder (z ranges from -half_length to +half_length)
    
    Returns
    -------
    entry_points : ndarray of shape (N, 3)
        Entry points into the cylinder (NaN for rays that miss)
    exit_points : ndarray of shape (N, 3)
        Exit points from the cylinder (NaN for rays that miss)
    """
    origins = np.atleast_2d(origins)
    directions = np.atleast_2d(directions)
    
    N = origins.shape[0]
    
    # Initialize output arrays with NaN
    entry_points = np.full((N, 3), np.nan)
    exit_points = np.full((N, 3), np.nan)
    
    # Extract components
    ox, oy, oz = origins[:, 0], origins[:, 1], origins[:, 2]
    dx, dy, dz = directions[:, 0], directions[:, 1], directions[:, 2]
    
    # --- Curved surface intersection ---
    # Solve (ox + t*dx)² + (oy + t*dy)² = R²
    # This is a quadratic: a*t² + b*t + c = 0
    a = dx**2 + dy**2
    b = 2 * (ox * dx + oy * dy)
    c = ox**2 + oy**2 - radius**2
    
    discriminant = b**2 - 4 * a * c
    
    # For each ray, we'll collect valid t values
    # We need to handle curved surface and end caps separately
    
    t_candidates = np.full((N, 4), np.inf)  # Up to 4 candidate t values per ray
    
    # Curved surface intersections (where discriminant >= 0 and a != 0)
    valid_curved = (discriminant >= 0) & (np.abs(a) > 1e-12)
    sqrt_disc = np.sqrt(np.maximum(discriminant, 0))
    
    t1 = np.where(valid_curved, (-b - sqrt_disc) / (2 * a), np.inf)
    t2 = np.where(valid_curved, (-b + sqrt_disc) / (2 * a), np.inf)
    
    # Check if curved surface intersections are within z bounds
    z1 = oz + t1 * dz
    z2 = oz + t2 * dz
    
    t1_valid = valid_curved & (t1 >= 0) & (np.abs(z1) <= half_length)
    t2_valid = valid_curved & (t2 >= 0) & (np.abs(z2) <= half_length)
    
    t_candidates[:, 0] = np.where(t1_valid, t1, np.inf)
    t_candidates[:, 1] = np.where(t2_valid, t2, np.inf)
    
    # --- End cap intersections ---
    # Top cap: z = +half_length, solve oz + t*dz = half_length
    # Bottom cap: z = -half_length
    
    # Avoid division by zero
    dz_safe = np.where(np.abs(dz) > 1e-12, dz, np.inf)
    
    t_top = (half_length - oz) / dz_safe
    t_bottom = (-half_length - oz) / dz_safe
    
    # Check if cap intersections are within radius
    x_top = ox + t_top * dx
    y_top = oy + t_top * dy
    x_bottom = ox + t_bottom * dx
    y_bottom = oy + t_bottom * dy
    
    r2_top = x_top**2 + y_top**2
    r2_bottom = x_bottom**2 + y_bottom**2
    
    t_top_valid = (t_top >= 0) & (r2_top <= radius**2) & (np.abs(dz) > 1e-12)
    t_bottom_valid = (t_bottom >= 0) & (r2_bottom <= radius**2) & (np.abs(dz) > 1e-12)
    
    t_candidates[:, 2] = np.where(t_top_valid, t_top, np.inf)
    t_candidates[:, 3] = np.where(t_bottom_valid, t_bottom, np.inf)
    
    # --- Find the two smallest positive t values (entry and exit) ---
    t_sorted = np.sort(t_candidates, axis=1)
    
    t_entry = t_sorted[:, 0]
    t_exit = t_sorted[:, 1]
    
    # Rays that hit have finite entry and exit times
    valid_hit = np.isfinite(t_entry) & np.isfinite(t_exit)
    
    # Compute intersection points
    entry_points[valid_hit] = origins[valid_hit] + t_entry[valid_hit, np.newaxis] * directions[valid_hit]
    exit_points[valid_hit] = origins[valid_hit] + t_exit[valid_hit, np.newaxis] * directions[valid_hit]
    
    return entry_points, exit_points

In [ ]:
# CMS-like dimensions (approximate, in meters)
cms_radius = 7.0        # ~7m radius
cms_half_length = 10.5  # ~21m total length

# Test with some muons coming from below
origins = np.array([
    [0, -20, 0],      # Directly below, pointing up
    [5, -15, 3],      # Off-center
    [100, -20, 0],    # Will miss (too far in x)
])

directions = np.array([
    [0, 1, 0],        # Straight up
    [-0.5, 1, -0.2],  # Angled
    [0, 1, 0],        # Straight up but will miss
])

entry, exit = ray_cylinder_intersection(origins, directions, cms_radius, cms_half_length)

print("Entry points:\n", entry)
print("Exit points:\n", exit)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

def visualize_ray_cylinder_intersections(
    origins: np.ndarray,
    directions: np.ndarray,
    entry_points: np.ndarray,
    exit_points: np.ndarray,
    radius: float,
    half_length: float,
    ax: plt.Axes = None,
    cylinder_alpha: float = 0.2,
    cylinder_color: str = 'cyan',
    ray_colors: list = None,
    figsize: tuple = (12, 10),
    extend_factor: float = 1.5,
) -> plt.Axes:
    """
    Visualize rays intersecting a cylinder.
    
    Parameters
    ----------
    origins : ndarray of shape (N, 3)
        Starting points of the rays
    directions : ndarray of shape (N, 3)
        Direction vectors of the rays
    entry_points : ndarray of shape (N, 3)
        Entry points into the cylinder (can contain NaN for misses)
    exit_points : ndarray of shape (N, 3)
        Exit points from the cylinder (can contain NaN for misses)
    radius : float
        Radius of the cylinder
    half_length : float
        Half-length of the cylinder
    ax : matplotlib 3D axes, optional
        Existing axes to plot on. If None, creates new figure.
    cylinder_alpha : float
        Transparency of the cylinder surface
    cylinder_color : str
        Color of the cylinder
    ray_colors : list, optional
        List of colors for each ray. If None, uses a colormap.
    figsize : tuple
        Figure size if creating new figure
    extend_factor : float
        How far to extend the dashed ray line beyond the cylinder
        
    Returns
    -------
    ax : matplotlib 3D axes
    """
    origins = np.atleast_2d(origins)
    directions = np.atleast_2d(directions)
    entry_points = np.atleast_2d(entry_points)
    exit_points = np.atleast_2d(exit_points)
    
    N = origins.shape[0]
    
    if ax is None:
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(111, projection='3d')
    
    # Generate ray colors if not provided
    if ray_colors is None:
        cmap = plt.cm.tab10
        ray_colors = [cmap(i % 10) for i in range(N)]
    
    # --- Draw the cylinder ---
    # Curved surface
    theta = np.linspace(0, 2 * np.pi, 50)
    z_cyl = np.linspace(-half_length, half_length, 30)
    theta_grid, z_grid = np.meshgrid(theta, z_cyl)
    x_cyl = radius * np.cos(theta_grid)
    y_cyl = radius * np.sin(theta_grid)
    
    ax.plot_surface(x_cyl, y_cyl, z_grid, alpha=cylinder_alpha, 
                    color=cylinder_color, edgecolor='none')
    
    # End caps
    r_cap = np.linspace(0, radius, 15)
    theta_cap = np.linspace(0, 2 * np.pi, 50)
    r_cap_grid, theta_cap_grid = np.meshgrid(r_cap, theta_cap)
    x_cap = r_cap_grid * np.cos(theta_cap_grid)
    y_cap = r_cap_grid * np.sin(theta_cap_grid)
    
    # Top cap
    z_top = np.full_like(x_cap, half_length)
    ax.plot_surface(x_cap, y_cap, z_top, alpha=cylinder_alpha, 
                    color=cylinder_color, edgecolor='none')
    
    # Bottom cap
    z_bottom = np.full_like(x_cap, -half_length)
    ax.plot_surface(x_cap, y_cap, z_bottom, alpha=cylinder_alpha, 
                    color=cylinder_color, edgecolor='none')
    
    # Draw cylinder wireframe edges for clarity
    theta_wire = np.linspace(0, 2 * np.pi, 50)
    ax.plot(radius * np.cos(theta_wire), radius * np.sin(theta_wire), 
            np.full_like(theta_wire, half_length), 'k-', alpha=0.3, linewidth=0.5)
    ax.plot(radius * np.cos(theta_wire), radius * np.sin(theta_wire), 
            np.full_like(theta_wire, -half_length), 'k-', alpha=0.3, linewidth=0.5)
    
    # --- Draw each ray ---
    for i in range(N):
        color = ray_colors[i]
        origin = origins[i]
        direction = directions[i]
        entry = entry_points[i]
        exit_pt = exit_points[i]
        
        # Check if this ray hit the cylinder
        hit = not np.any(np.isnan(entry))
        
        # Draw origin point
        ax.scatter(*origin, color=color, s=100, marker='o', 
                   edgecolors='black', linewidths=1, label=f'Ray {i+1} origin' if i < 5 else None)
        
        if hit:
            # Draw entry and exit points
            ax.scatter(*entry, color=color, s=50, marker='o', 
                       edgecolors='black', linewidths=1.5, zorder=5)
            ax.scatter(*exit_pt, color=color, s=50, marker='o', 
                       edgecolors='black', linewidths=1.5, zorder=5)
            
            # Draw solid line through cylinder (entry to exit)
            ax.plot([entry[0], exit_pt[0]], 
                    [entry[1], exit_pt[1]], 
                    [entry[2], exit_pt[2]], 
                    color=color, linewidth=2.5, solid_capstyle='round')
            
            # Calculate how far to extend the dashed line
            dir_norm = direction / np.linalg.norm(direction)
            
            # Dashed line from origin to entry
            ax.plot([origin[0], entry[0]], 
                    [origin[1], entry[1]], 
                    [origin[2], entry[2]], 
                    color=color, linewidth=1.5, linestyle='--', alpha=0.7)
            
            # Dashed line extending beyond exit
            extend_dist = extend_factor * np.linalg.norm(exit_pt - entry)
            end_point = exit_pt + dir_norm * extend_dist
            ax.plot([exit_pt[0], end_point[0]], 
                    [exit_pt[1], end_point[1]], 
                    [exit_pt[2], end_point[2]], 
                    color=color, linewidth=1.5, linestyle='--', alpha=0.7)
        else:
            # Ray missed - draw extended dashed line
            dir_norm = direction / np.linalg.norm(direction)
            extend_dist = 4 * half_length  # Extend far enough to show it misses
            end_point = origin + dir_norm * extend_dist
            ax.plot([origin[0], end_point[0]], 
                    [origin[1], end_point[1]], 
                    [origin[2], end_point[2]], 
                    color=color, linewidth=1.5, linestyle=':', alpha=0.5)
    
    # --- Set labels and adjust view ---
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    
    # Set equal aspect ratio
    max_range = max(radius, half_length) * 1.5
    
    # Include origins in the range calculation
    all_points = [origins]
    if not np.all(np.isnan(entry_points)):
        all_points.append(entry_points[~np.isnan(entry_points).any(axis=1)])
    if not np.all(np.isnan(exit_points)):
        all_points.append(exit_points[~np.isnan(exit_points).any(axis=1)])
    
    all_points = np.vstack(all_points)
    
    x_range = [min(all_points[:, 0].min(), -radius) - 2, 
               max(all_points[:, 0].max(), radius) + 2]
    y_range = [min(all_points[:, 1].min(), -radius) - 2, 
               max(all_points[:, 1].max(), radius) + 2]
    z_range = [min(all_points[:, 2].min(), -half_length) - 2, 
               max(all_points[:, 2].max(), half_length) + 2]
    
    ax.set_xlim(x_range)
    ax.set_ylim(y_range)
    ax.set_zlim(z_range)
    
    # Try to set equal aspect ratio (works in newer matplotlib)
    try:
        ax.set_aspect('equal')
    except NotImplementedError:
        pass
    
    ax.set_title('Muon Trajectories through CMS Detector')
    
    return ax


def demo_visualization():
    """Run a demonstration of the ray-cylinder intersection and visualization."""
    
    # CMS-like dimensions (in meters)
    cms_radius = 7.0
    cms_half_length = 10.5
    
    # Create test muons coming from underground (negative y)
    np.random.seed(42)
    
    origins = np.array([
        [0, -20, 0],           # Directly below center
        [3, -18, 5],           # Off-center
        [-2, -22, -3],         # Another off-center
        [8, -15, 0],           # Near edge
        [20, -20, 0],          # Will miss (too far in x)
        [0, -25, 15],          # Will miss (too far in z)
    ])
    
    # Direction vectors (roughly pointing upward with some spread)
    directions = np.array([
        [0, 1, 0],             # Straight up
        [-0.1, 1, -0.2],       # Slight angle
        [0.15, 1, 0.1],        # Slight angle
        [-0.3, 1, 0],          # Angled toward center
        [0, 1, 0],             # Straight up (but will miss)
        [0, 1, -0.3],          # Angled (but will miss)
    ])
    
    # Compute intersections
    entry_points, exit_points = ray_cylinder_intersection(
        origins, directions, cms_radius, cms_half_length
    )
    
    print("Entry points:")
    print(entry_points)
    print("\nExit points:")
    print(exit_points)
    
    # Visualize
    ax = visualize_ray_cylinder_intersections(
        origins, directions, entry_points, exit_points,
        cms_radius, cms_half_length
    )
    ax.view_init(vertical_axis='y')
    ax.set_xlim(-20,20)
    ax.set_ylim(-20,20)
    ax.set_zlim(-20,20)

    plt.tight_layout()
    plt.show()

    return ax


#if __name__ == "__main__":
ax = demo_visualization()




In [ ]:
np.tan(np.deg2rad(91))